# Probabilities

In [8]:
from pathlib import Path
import json
import re

import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_ROOT = PROJECT_ROOT / "data" / "cut_images" / "split" / "test"

with open(PROJECT_ROOT / "data" / "mapping.json", "r", encoding="utf-8") as f:
    mapping = json.load(f)["id_to_name"]

# Keep the defined habitat names in a fixed order.
habitats = ["dense", "dense_leafless", "dense_snow", "open", "open_leafless", "open_snow", "sparse", "sparse_leafless", "sparse_snow"]
species_order = [mapping[key] for key in sorted(mapping)]

# Cutout filenames look like: flight_frame_g<cluster>_<count><species><count><species>...jpg
species_chunk_pattern = re.compile(r"(\d+)([a-l])", re.IGNORECASE)
filename_suffix_pattern = re.compile(r"_g\d+_(.+)$")

records: list[dict[str, object]] = []
skipped_files: list[Path] = []



In [9]:
for image_path in DATA_ROOT.rglob("*.jpg"):
    habitat = image_path.parent.name
    suffix_match = filename_suffix_pattern.search(image_path.stem)
    if habitat not in habitats or suffix_match is None:
        skipped_files.append(image_path)
        continue

    species_suffix = suffix_match.group(1)
    species_chunks = species_chunk_pattern.findall(species_suffix)
    if not species_chunks:
        skipped_files.append(image_path)
        continue

    for count_text, class_id in species_chunks:
        records.append(
            {
                "habitat": habitat,
                "species": mapping[class_id.lower()],
                "count": int(count_text),
            }
        )
    


counts_df = pd.DataFrame(records)
if counts_df.empty:
    raise ValueError(f"No parseable cutout filenames found under {DATA_ROOT}")

joint_counts = counts_df.groupby(["habitat", "species"], as_index=False)["count"].sum()

joint_pivot = (
    joint_counts.pivot(index="habitat", columns="species", values="count")
    .reindex(index=habitats, columns=species_order)
    .fillna(0)
)

p_species_given_habitat = joint_pivot.div(joint_pivot.sum(axis=1), axis=0).fillna(0)
p_habitat_given_species = joint_pivot.T.div(joint_pivot.T.sum(axis=1), axis=0).fillna(0)


In [10]:
# Display probabilities as percentages for readability.
print("P(species | habitat) [%]")
display((p_species_given_habitat * 100).round(1))

print("P(habitat | species) [%]")
display((p_habitat_given_species * 100).round(1))

if skipped_files:
    print(f"Skipped {len(skipped_files)} files that did not match the expected habitat/species filename pattern.")

non_zero_species = (joint_pivot.sum(axis=0) > 0).sum()
if non_zero_species == 1:
    print("Note: the current cutout data contains only one species, so the remaining probabilities are 0.0%.")

P(species | habitat) [%]


species,No-animal,Cervus elaphus (Red deer),Capreolus capreolus (Roe deer),Rupicapra rupicapra (Chamois),Homo sapiens (Human),Capra ibex (Alpine ibex),Dama dama (Fallow Deer),Unknown,Canis lupus familiaris (Dog),Aves (Bird),Sus scrofa (Wild boar),Sus scrofa x Sus domesticus (Hybrid Pig)
habitat,,,,,,,,,,,,
dense,0.0,30.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,69.1,0.0
dense_leafless,0.0,75.3,6.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.8,3.7
dense_snow,0.0,0.0,11.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,88.9,0.0
open,0.0,81.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,0.0,17.4,1.5
open_leafless,0.0,44.1,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,32.8,16.1
open_snow,0.0,42.2,57.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
sparse,0.0,63.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,36.8,0.0
sparse_leafless,0.0,47.2,1.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,36.8,14.3
sparse_snow,0.0,83.1,16.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


P(habitat | species) [%]


habitat,dense,dense_leafless,dense_snow,open,open_leafless,open_snow,sparse,sparse_leafless,sparse_snow
species,,,,,,,,,
No-animal,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cervus elaphus (Red deer),2.1,4.2,0.0,30.5,19.6,3.4,18.2,18.4,3.7
Capreolus capreolus (Roe deer),0.0,3.6,0.7,0.0,32.4,48.2,0.0,7.2,7.9
Rupicapra rupicapra (Chamois),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Homo sapiens (Human),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Capra ibex (Alpine ibex),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Dama dama (Fallow Deer),0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0
Unknown,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Canis lupus familiaris (Dog),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
